# Predykcyjne kodowanie mowy


In [ ]:
from pathlib import Path
import struct

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
from IPython.display import Audio, display


### Wczytanie nagrania

Plik `mowa.wav` zawiera 20 sekund mowy w formacie PCM 16-bit, mono, 11 025 Hz.


In [ ]:
probki, fs = sf.read("mowa.wav", dtype="int16")
print(f"Próbkowanie: {fs} Hz, czas: {len(probki) / fs:.2f} s")


### Przebieg i odsłuch


In [ ]:
x = np.asarray(probki, dtype=np.float64) / 32768
t = np.arange(len(x)) / fs

fig, ax = plt.subplots(figsize=(11, 3), constrained_layout=True)
ax.plot(t, x, lw=0.5)
ax.set(title="Nagranie wejściowe", xlabel="Czas [s]", ylabel="Amplituda")
ax.grid(alpha=0.3)
plt.show()

display(Audio(x, rate=fs, normalize=False))


## Segmentacja

Segment ma 256 próbek. Sąsiednie segmenty mają 10 wspólnych próbek, więc ich początki dzieli 246 próbek. Końcówkę ostatniego segmentu uzupełniamy zerami.


In [ ]:
N = 256
r = 10
krok = N - r

liczba_probek = len(probki)
liczba_segmentow = 1 + (max(0, liczba_probek - N) + krok - 1) // krok
poczatki = np.arange(liczba_segmentow) * krok
liczba_zer = poczatki[-1] + N - liczba_probek
probki_z_dopelnieniem = np.pad(probki, (0, liczba_zer))
segmenty = probki_z_dopelnieniem[poczatki[:, None] + np.arange(N)]

print(f"Segmenty: {len(segmenty)}, próbki na segment: {N}, zera na końcu: {liczba_zer}")


### Pierwsze dwa segmenty


In [ ]:
fig, ax = plt.subplots(figsize=(11, 3), constrained_layout=True)
for i in range(2):
    indeksy = poczatki[i] + np.arange(N)
    ax.plot(indeksy, segmenty[i] / 32768, lw=0.8, label=f"Segment {i + 1}")

ax.axvspan(krok - 0.5, N - 0.5, alpha=0.2, color="gray", label="Wspólne próbki")
ax.set(xlabel="Numer próbki (od 0)", ylabel="Amplituda")
ax.grid(alpha=0.3)
ax.legend()
plt.show()


## AR(10) i algorytm Levinsona–Durbina

Przed obliczeniem autokorelacji mnożymy segment przez okno Hanna i dodajemy po 10 zer z obu stron.

$$w(k)=\frac12\left(1-\cos\frac{2\pi k}{N+1}\right),\qquad k=1,\ldots,N.$$

Współczynniki wyznaczamy osobno dla każdego segmentu. Przyjmujemy znak zgodny z instrukcją.

$$e(k)=y(k)+\sum_{i=1}^{r} a_i\,y(k-i).$$


In [ ]:
def levinson_durbin(autokorelacja, rzad):
    wspolczynniki = np.zeros(rzad)
    energia = autokorelacja[0]

    if energia == 0:
        return wspolczynniki

    for aktualny_rzad in range(1, rzad + 1):
        poprzednie = wspolczynniki[:aktualny_rzad - 1].copy()

        licznik = autokorelacja[aktualny_rzad]
        for i in range(aktualny_rzad - 1):
            licznik += poprzednie[i] * autokorelacja[aktualny_rzad - 1 - i]

        nowy_wspolczynnik = -licznik / energia

        for i in range(aktualny_rzad - 1):
            wspolczynniki[i] = poprzednie[i] + nowy_wspolczynnik * poprzednie[aktualny_rzad - 2 - i]

        wspolczynniki[aktualny_rzad - 1] = nowy_wspolczynnik 
        energia *= 1 - nowy_wspolczynnik**2

    return wspolczynniki


okno = np.hanning(N + 2)[1:-1]
segmenty_okienkowane = segmenty * okno
autokorelacje = []

for segment in segmenty_okienkowane:
    segment_z_zerami = np.pad(segment, (r, r))
    korelacja = np.correlate(segment_z_zerami, segment, mode="valid")
    autokorelacje.append(korelacja[r:])

autokorelacje = np.array(autokorelacje)

parametry_ar = np.array([levinson_durbin(R, r) for R in autokorelacje])
print(f"Modele AR({r}): {len(parametry_ar)}")


### Przebieg współczynników AR(10)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4), constrained_layout=True)
ax.plot(poczatki / fs, parametry_ar, lw=0.7)
ax.set(xlabel="Czas [s]", ylabel="Wartość współczynnika")
ax.grid(alpha=0.3)
ax.legend([f"a{i}" for i in range(1, r + 1)], ncol=5)
plt.show()


## Błąd predykcji

Współczynniki AR stosujemy do próbek PCM bez okna Hanna. Na początku nagrania brakujące próbki historii zastępujemy zerami.

$$e(k)=y(k)+\sum_{i=1}^{r}a_i\,y(k-i).$$


In [ ]:
probki_z_historia = np.pad(probki_z_dopelnieniem, (r, 0))
indeksy_probek = poczatki[:, None] + np.arange(N) + r
bledy = segmenty.astype(float)

for opoznienie in range(1, r + 1):
    poprzednie_probki = probki_z_historia[indeksy_probek - opoznienie]
    a = parametry_ar[:, opoznienie - 1, None]
    bledy += a * poprzednie_probki

e_max = np.max(np.abs(bledy), axis=1)


## Kwantyzacja błędu

Dla każdego segmentu rozmieszczamy $2^m$ poziomów równomiernie od $-e_{\max}$ do $e_{\max}$. Każdy błąd zastępujemy numerem najbliższego poziomu.


In [ ]:
def kwantyzuj(bledy, e_max, bity):
    kody = np.zeros(bledy.shape, dtype=np.uint8)

    for i, blad in enumerate(bledy):
        if e_max[i] > 0:
            poziomy = 2**bity - 1
            indeks = (blad / e_max[i] + 1) * poziomy / 2
            kody[i] = np.rint(indeks)

    return kody


warianty = (2, 3, 4)
kody = {}
for bity in warianty:
    kody[bity] = kwantyzuj(bledy, e_max, bity)


### Błąd przed i po kwantyzacji


In [ ]:
i = np.random.randint(len(segmenty))
fig, osie = plt.subplots(3, 1, figsize=(11, 7), sharex=True, sharey=True,
                         constrained_layout=True)

for ax, bity in zip(osie, warianty):
    blad_po_kwantyzacji = e_max[i] * (2 * kody[bity][i] / (2**bity - 1) - 1)
    ax.plot(bledy[i], lw=0.8, label="Błąd predykcji")
    ax.step(np.arange(N), blad_po_kwantyzacji, where="mid", lw=0.8,
            label=f"Po kwantyzacji ({bity} bity)")
    ax.set_ylabel("Amplituda [PCM]")
    ax.grid(alpha=0.3)
    ax.legend()

fig.suptitle(f"Losowy segment {i + 1}")
osie[-1].set_xlabel("Numer próbki w segmencie")
plt.show()


## Zapis binarny

Nagłówek zawiera identyfikator formatu, częstotliwość próbkowania, liczbę próbek wejścia, długość segmentu, rząd modelu i liczbę bitów kwantyzacji. Po nagłówku zapisujemy dla każdego segmentu współczynniki AR, maksymalny błąd i 256 kodów.

Współczynniki oraz maksymalny błąd zapisujemy jako liczby 64-bitowe. Każdy kod zajmuje dokładnie tyle bitów, ile przewiduje dany wariant. Bity trafiają do bajtów od najbardziej znaczącego.


In [ ]:
def spakuj_kody(kody_segmentu, bity):
    bajty = bytearray()
    bufor = 0
    zajete_bity = 0

    for kod in kody_segmentu:
        bufor = (bufor << bity) | int(kod)
        zajete_bity += bity

        if zajete_bity >= 8:
            zajete_bity -= 8
            bajty.append(bufor >> zajete_bity)
            bufor &= (1 << zajete_bity) - 1

    if zajete_bity:
        bajty.append(bufor << (8 - zajete_bity))

    return bajty


In [ ]:
format_naglowka = struct.Struct("<4sIIHBB")
format_segmentu = struct.Struct(f"<{r + 1}d")


def zapisz_kodowanie(sciezka, bity):
    with sciezka.open("wb") as plik:
        plik.write(format_naglowka.pack(b"PAR1", fs, liczba_probek, N, r, bity))

        for a, maksymalny_blad, kody_segmentu in zip(parametry_ar, e_max, kody[bity]):
            plik.write(format_segmentu.pack(*a, maksymalny_blad))
            plik.write(spakuj_kody(kody_segmentu, bity))


pliki_kodowania = {}
for bity in warianty:
    sciezka = Path(f"mowa_{bity}bit.bin")
    zapisz_kodowanie(sciezka, bity)
    pliki_kodowania[bity] = sciezka


### Stopień kompresji


In [ ]:
rozmiar_wav = Path("mowa.wav").stat().st_size
print(f"WAV: {rozmiar_wav} B")
print(f"{'Bity':>4} {'Rozmiar [B]':>12} {'WAV / BIN':>10} {'Redukcja':>10}")

for bity, sciezka in pliki_kodowania.items():
    rozmiar = sciezka.stat().st_size
    kompresja = rozmiar_wav / rozmiar
    redukcja = 1 - rozmiar / rozmiar_wav
    print(f"{bity:>4} {rozmiar:>12} {kompresja:>9.2f}× {redukcja:>9.1%}")


## Dekoder

Z pliku odczytujemy współczynniki AR, maksymalny błąd i kody. Każdy kod zamieniamy na wartość błędu i dodajemy ją do prognozy z poprzednich odtworzonych próbek.

$$\hat y(k)=\hat e(k)-\sum_{i=1}^{r}a_i\,\hat y(k-i).$$

Pierwszy segment zaczyna z zerową historią. W kolejnych segmentach pierwsze 10 próbek to nakładka z poprzednim segmentem. Te próbki są już odtworzone i służą jako historia do rekonstrukcji pozostałych 246 próbek.


In [ ]:
def rozpakuj_kody(dane, bity, liczba_kodow):
    bajty = np.frombuffer(dane, dtype=np.uint8)
    zapisane_bity = np.unpackbits(bajty)[:liczba_kodow * bity]
    grupy = zapisane_bity.reshape(liczba_kodow, bity)
    wagi_bitow = 2**np.arange(bity - 1, -1, -1)
    return grupy @ wagi_bitow


In [ ]:
def odczytaj_kodowanie(sciezka):
    with sciezka.open("rb") as plik:
        identyfikator, fs, liczba_probek, N, r, bity = format_naglowka.unpack(
            plik.read(format_naglowka.size)
        )
        krok = N - r
        liczba_segmentow = 1 + (max(0, liczba_probek - N) + krok - 1) // krok
        format_segmentu = struct.Struct(f"<{r + 1}d")
        rozmiar_kodow = (N * bity + 7) // 8
        odtworzone = np.zeros(r + N + (liczba_segmentow - 1) * krok)

        for i in range(liczba_segmentow):
            dane_segmentu = format_segmentu.unpack(plik.read(format_segmentu.size))
            a = np.array(dane_segmentu[:-1])
            maksymalny_blad = dane_segmentu[-1]
            kody_segmentu = rozpakuj_kody(plik.read(rozmiar_kodow), bity, N)
            bledy_segmentu = maksymalny_blad * (2 * kody_segmentu / (2**bity - 1) - 1)
            poczatek = r + i * krok
            pierwsza_nowa = 0 if i == 0 else r

            for j in range(pierwsza_nowa, N):
                pozycja = poczatek + j
                historia = odtworzone[pozycja - r:pozycja][::-1]
                odtworzone[pozycja] = bledy_segmentu[j] - a @ historia

    return odtworzone[r:r + liczba_probek], fs


### Zapis odtworzonych nagrań


In [ ]:
pliki_wav = {}
for bity, sciezka in pliki_kodowania.items():
    odtworzone, fs_wyniku = odczytaj_kodowanie(sciezka)
    pcm = np.clip(np.rint(odtworzone), -32768, 32767).astype(np.int16)
    nazwa = Path(f"mowa_{bity}bit_odtworzona.wav")
    sf.write(nazwa, pcm, fs_wyniku, subtype="PCM_16")
    pliki_wav[bity] = nazwa
    print(f"{nazwa} – {len(pcm) / fs_wyniku:.2f} s")


## Porównanie rekonstrukcji

Na wykresach pokazujemy ten sam fragment wokół próbki o największej amplitudzie w oryginale. SNR mierzymy na całym nagraniu względem wejściowego WAV. Wyższy wynik oznacza mniejszy błąd rekonstrukcji.


In [ ]:
nagrania = {"Oryginał": x}
print(f"{'Bity':>4} {'MAE [PCM]':>12} {'SNR [dB]':>10}")

for bity, sciezka in pliki_wav.items():
    pcm, _ = sf.read(sciezka, dtype="int16")
    nagrania[f"{bity} bity"] = pcm / 32768

    roznica = probki.astype(float) - pcm
    mae = np.mean(np.abs(roznica))
    snr = 10 * np.log10(np.sum(x**2) / np.sum((roznica / 32768)**2))
    print(f"{bity:>4} {mae:>12.1f} {snr:>10.2f}")


### Przebiegi sygnałów


In [ ]:
srodek = np.argmax(np.abs(x))
polowa = int(0.025 * fs)
fragment = slice(max(0, srodek - polowa), srodek + polowa)
fig, osie = plt.subplots(len(nagrania), 1, figsize=(11, 7),
                         sharex=True, sharey=True, constrained_layout=True)

for ax, (nazwa, sygnal) in zip(osie, nagrania.items()):
    ax.plot(t[fragment] * 1000, sygnal[fragment], lw=0.8)
    ax.set(title=nazwa, ylabel="Amplituda")
    ax.grid(alpha=0.3)

osie[-1].set_xlabel("Czas [ms]")
plt.show()


### Odsłuch


In [ ]:
for nazwa, sygnal in nagrania.items():
    print(nazwa)
    display(Audio(sygnal, rate=fs, normalize=False))
